# Phase 4 Live Demo — Explainability & Polymer Invariance
This notebook accompanies the Phase 4 trustworthiness report. It uses the proxy ensemble trained by `01_proxy_models.py` and the artifacts in `outputs/`.

In [ ]:
import sys, os
sys.path.insert(0, 'scripts')
import numpy as np, pandas as pd
from helpers import (OUTPUT_DIR, load_train, load_test, load_proxy,
                     rebuild_features, predict_ensemble, random_smiles,
                     canonical_smiles, featurize)
train = load_train()
test = load_test()

## 1. One polymer, five equivalent SMILES forms

In [ ]:
pkl = load_proxy('tg')
row = train[train['target_type'] == 'tg'].iloc[0]
variants = [row['smiles']] + random_smiles(row['smiles'], 4)
df = pd.DataFrame({'smiles': variants,
                   'canonical': [canonical_smiles(v) for v in variants]})
X, _ = featurize(variants, pipe=pkl['pipe'], canonicalize=False)
df['pred_tg'] = predict_ensemble(X, pkl)
df

## 2. Predictions are identical (invariance)

In [ ]:
print('prediction std across 5 SMILES forms: {:.5f} K'.format(df['pred_tg'].std()))
assert df['pred_tg'].std() < 1.0, 'representation invariance holds'

## 3. Confidence interval (split-conformal)

In [ ]:
intervals = pd.read_csv(OUTPUT_DIR / 'test_predictions_with_intervals.csv')
intervals.head(5)

## 4. Reliability tier (AD similarity x uncertainty)

In [ ]:
tiers = pd.read_csv(OUTPUT_DIR / 'reliability_tiers_test.csv')
tiers['tier'].value_counts()

## 5. Counterfactual: minimal change to raise Tg by +20 K

In [ ]:
cf = pd.read_csv(OUTPUT_DIR / 'counterfactual_directions_tg.csv')
cf.head()